# 👀 Multilayer perceptron (MLP)

In this notebook, we'll walk through the steps required to train your own multilayer perceptron on the CIFAR dataset using PyTorch

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

from notebooks.utils_pytorch import display

In [ ]:
# Select device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
NUM_CLASSES = 10
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.0005

## 1. Prepare the Data <a name="prepare"></a>

In [ ]:
train_data = datasets.CIFAR10(root=REPO_ROOT / "data", train=True, download=True)
test_data = datasets.CIFAR10(root=REPO_ROOT / "data", train=False, download=True)

In [ ]:
# NHWC uint8 -> NCHW float32 in [0, 1]
x_train = torch.tensor(train_data.data).permute(0, 3, 1, 2).float() / 255.0
x_test = torch.tensor(test_data.data).permute(0, 3, 1, 2).float() / 255.0

# nn.CrossEntropyLoss takes class indices, so no one-hot encoding is needed
y_train = torch.tensor(train_data.targets)
y_test = torch.tensor(test_data.targets)

train_loader = DataLoader(
    TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(x_test, y_test), batch_size=BATCH_SIZE, shuffle=False
)

In [ ]:
display(x_train[:10])
print(y_train[:10])

## 2. Build the model <a name="build"></a>

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 200),
            nn.ReLU(),
            nn.Linear(200, 150),
            nn.ReLU(),
            # Outputs logits - the softmax is applied inside nn.CrossEntropyLoss
            nn.Linear(150, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = MLP(NUM_CLASSES).to(device)

print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Train the model <a name="train"></a>

In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train_one_epoch(model, loader, loss_fn, opt):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()

        total_loss += loss.item() * len(x)
        correct += (logits.argmax(dim=-1) == y).sum().item()
        total += len(x)
    return total_loss / total, correct / total


for epoch in range(EPOCHS):
    loss, acc = train_one_epoch(model, train_loader, loss_fn, opt)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {loss:.4f} - accuracy: {acc:.4f}")

## 4. Evaluation <a name="evaluate"></a>

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(x)
        correct += (logits.argmax(dim=-1) == y).sum().item()
        total += len(x)
    return total_loss / total, correct / total


test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} - Test accuracy: {test_acc:.4f}")

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    return torch.cat(
        [torch.softmax(model(x.to(device)), dim=-1).cpu() for x, _ in loader]
    ).numpy()


preds = predict(model, test_loader)
preds_single = CLASSES[np.argmax(preds, axis=-1)]
actual_single = CLASSES[y_test.numpy()]

In [ ]:
n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx].permute(1, 2, 0).numpy()  # CHW -> HWC for imshow
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)